# Repository execution setup
Use Python 3.10/3.11. For QLoRA select a CUDA GPU before installing. This notebook calls repository scripts; training code lives in `src/llm_grading/`.
Do not upload private student data to a public notebook. Keep outputs private and use persistent storage if the runtime may disconnect.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json
REPO = Path.cwd() if (Path.cwd() / "scripts/train.py").exists() else Path("/content/LLM-APA" if Path("/content").exists() else "/kaggle/working/LLM-APA")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/duykhang2103/LLM-APA.git", str(REPO)], check=True)
os.chdir(REPO)
# Notebook kernels may use another Python version; repository scripts use Python 3.11.
subprocess.run([sys.executable, "-m", "pip", "install", "uv", "pyyaml"], check=True)
notebook_env = REPO / ".notebook-venv"
setup_env = dict(os.environ, UV_PROJECT_ENVIRONMENT=str(notebook_env))
subprocess.run([sys.executable, "-m", "uv", "sync", "--python", "3.11", "--group", "dev", "--group", "ml"], cwd=REPO, env=setup_env, check=True)
PYTHON = str(notebook_env / ("Scripts/python.exe" if os.name == "nt" else "bin/python"))
def run(script, *args):
    subprocess.run([PYTHON, str(REPO / "scripts" / script), *map(str, args)], cwd=REPO, check=True)


## Optional persistent storage
Set `PERSISTENT` below to a mounted volume, Google Drive, or a Kaggle output directory. On Colab you may mount Drive with the next cell. This is optional; no cloud SDK is used by the training code. Kaggle `/kaggle/working` must be exported/saved before ending the session.


In [ ]:
# Optional Colab only; uncomment if needed.
# from google.colab import drive
# drive.mount("/content/drive")
PERSISTENT = Path("/content/drive/MyDrive/LLM-APA-runs") if Path("/content/drive/MyDrive").exists() else REPO / "outputs/notebook-runs"
PERSISTENT.mkdir(parents=True, exist_ok=True)


# Zero-shot / RAG experiments
P0 has no demonstrations. Freeze its prompt/model/settings, then switch to P1 training-only RAG. Prefer deterministic decoding and use the same validation IDs/evaluator as fine-tuning. Local Qwen defaults require CUDA; the API adapter uses an environment-variable key and explicit course data permission.


In [ ]:
import yaml
TASK = "task1"  # task1 / task2 / task3
DATA_ROOT = Path("/content/private-data")  # change to supplied teacher-format directory
TRAIN_FILE = None   # optional official train file/directory
VAL_FILE = None     # official dev file/directory; set BOTH train and val when supplied
TEST_FILE = None    # lecturer test file/directory, if supplied
METHOD = "zero_shot"  # zero_shot / rag
VERSION = "P0-v1"     # P1-v1 for RAG; new version for each hypothesis
K = 3                # RAG supports 1, 3, 5
EXCLUDE_DUPLICATES = True  # compare False vs True in separate versions
config = yaml.safe_load((REPO / f"configs/{TASK}/{METHOD}.yaml").read_text())
config["experiment"].update(id=f"{TASK}-{VERSION}", notes="baseline")
config["data"].update(input_path=str(DATA_ROOT), train_path=TRAIN_FILE, validation_path=VAL_FILE, test_path=TEST_FILE,
                      prepared_dir=str(PERSISTENT / TASK / "prepared-v1"))
config["saving"]["output_dir"] = str(PERSISTENT / TASK / VERSION)
if METHOD == "rag":
    config["retrieval"].update(k=K, exclude_duplicates=EXCLUDE_DUPLICATES)
# API alternative (only if sending student data is permitted):
# import getpass
# os.environ["LLM_API_KEY"] = getpass.getpass("API key: ")
# config["model"] = {"provider":"api", "name_or_path":"provider-model", "base_url":"https://provider/v1", "api_key_env":"LLM_API_KEY", "allow_private_data":True}
CONFIG = PERSISTENT / f"{TASK}-{VERSION}-config.json"
CONFIG.write_text(json.dumps(config, indent=2))
run("prepare_data.py", "--config", CONFIG, "--training")
PREPARED = Path(config["data"]["prepared_dir"]) / ("config-" + config["experiment"]["id"] + ".json")


## Small smoke subset, then full validation
Smoke artifacts live under `smoke/`. RAG logs candidate exclusions/retrieved IDs; Task 1/2 examples exclude feedback. Task 3 filters to the requested level before ranking.


In [ ]:
run("smoke.py", "--config", PREPARED)
run("predict.py", "--config", PREPARED, "--split", "val")
OUT = Path(config["saving"]["output_dir"])
run("validate_predictions.py", "--task", TASK, "--input", OUT / "predictions.json")
run("evaluate.py", "--config", PREPARED, "--split", "val", "--predictions", OUT / "predictions.json")


## Inspect metrics and responses
Keep artifacts private. Raw model responses may quote student code. Macro-F1 uses all ten labels with zero_division=0; compare per-label support. Task 3 overlap/compliance are diagnostics requiring content review.


In [ ]:
metrics = json.loads((OUT / "metrics.json").read_text())
predictions = json.loads((OUT / "predictions.json").read_text())
print(json.dumps(metrics, ensure_ascii=False, indent=2))
predictions[:3]


## Final acceptance
Freeze settings before lecturer test data arrives. Configure its path and run prediction → validation → packaging without target inspection or retraining.


In [ ]:
# run("predict.py", "--config", PREPARED, "--input", TEST_FILE, "--output", OUT / "final_predictions.json")
# run("package_predictions.py", "--task", TASK, "--input", OUT / "final_predictions.json", "--output", OUT / "submission.zip")
